In [ ]:
# COLORS       = {'Eastern Arabian Sea':'blood',
#                 'Central India':'#D42028',
#                 'Central Bay of Bengal':'#F2C85E',
#                 'Equatorial Indian Ocean':'#5BA7DA',
#                 'Konkan Coast':'#1B2C61'}

In [2]:
import os
import json
import warnings
import numpy as np
import pandas as pd
import xarray as xr
import sympy as sp
import proplot as pplt
warnings.filterwarnings('ignore')
pplt.rc.update({'figure.dpi':100,'reso':'xx-hi'})

In [ ]:
with open('../scripts/configs.json','r',encoding='utf-8') as f:
    CONFIGS = json.load(f)
SPLITSDIR  = CONFIGS['filepaths']['splits']
MODELSDIR  = CONFIGS['filepaths']['models']
PREDSDIR   = CONFIGS['filepaths']['predictions']
MODELS     = CONFIGS['experiments']
SRCONFIG   = MODELS['sr']
SPLIT      = 'valid'
SEEDS      = SRCONFIG['seeds']
PARETO_CS  = [4,6,8,10,11,13,16,18,20,22,24]
PLOTSTYLE  = {
    'pod':           ('blood',   'POD'),
    'baseline':      ('#5BA7DA', 'Baseline NN'),
    'nonparametric': ('#F2C85E', 'Nonparametric Kernel NN'),
    'parametric':    ('#D42028', 'Parametric Kernel NN'),
    'sr':            ('#D42028', 'SR Pareto'),
    'sr_opt':        ('#1B2C61', 'Optimized SR Equation'),
}

In [ ]:
def get_kind(name):
    for group in ['pod','nn','sr']:
        runs = MODELS.get(group,{}).get('runs',{})
        if name in runs:
            if group in ('pod','sr'):
                return group
            return runs[name]['kind']
    if name in SRCONFIG.get('optimizedeqs',{}):
        return 'sr_opt'
    return 'unknown'

def get_r2(ytrue,ypred,dims=None):
    dims  = list(ytrue.dims) if dims is None else dims
    ssres = ((ytrue-ypred)**2).sum(dim=dims,skipna=True)
    sstot = ((ytrue-ytrue.mean(dim=dims,skipna=True))**2).sum(dim=dims,skipna=True)
    return 1-ssres/sstot

def get_mse(ytrue,ypred,dims=None):
    dims = list(ytrue.dims) if dims is None else dims
    return ((ytrue-ypred)**2).mean(dim=dims,skipna=True)

def get_nn_complexity(kind,nfieldvars,nlevs,nlocalvars):
    def nparams(nfeatures):
        return (nfeatures*256)+256+(256*128)+128+(128*64)+64+(64*32)+32+(32*1)+1
    if kind=='baseline':
        return nparams(nfieldvars*nlevs+nlocalvars)
    elif kind=='nonparametric':
        return nfieldvars*nlevs+nparams(nfieldvars+nlocalvars)
    elif kind=='parametric':
        return 2*nfieldvars+nparams(nfieldvars+nlocalvars)

def get_mse_at_r2(ytrue,r2_target=0.5,dims=None):
    dims  = list(ytrue.dims) if dims is None else dims
    sstot = ((ytrue-ytrue.mean(dim=dims,skipna=True))**2).sum(dim=dims,skipna=True)
    n     = ytrue.count(dim=dims)
    return (1-r2_target)*sstot/n

def pareto_front(records):
    ordered      = sorted(records,key=lambda r: r['mse'])
    front        = []
    best_nparams = np.inf
    for r in ordered:
        if r['nparams'] < best_nparams:
            front.append(r)
            best_nparams = r['nparams']
    return front

def prettify_eq(eq_str):
    all_vars = ['rh','thetae','thetaestar','lf','shf','lhf']
    var_syms = {name: sp.Symbol(name) for name in all_vars}
    symbol_names = {
        var_syms['rh']:         r'\widehat{RH}',
        var_syms['thetae']:     r'\widehat{\theta_e}',
        var_syms['thetaestar']: r'\widehat{\theta_e^*}',
        var_syms['lf']:         r'\mathrm{LF}',
        var_syms['shf']:        r'\mathrm{SHF}',
        var_syms['lhf']:        r'\mathrm{LHF}'}
    ns = {**var_syms,
          'cube':lambda x:x**3,'square':lambda x:x**2,'neg':lambda x:-x,
          'sqrt':sp.sqrt,'exp':sp.exp,'log':sp.log,'abs':sp.Abs,'cos':sp.cos,'sin':sp.sin}
    try:
        expr = eval(eq_str,{'__builtins__':{}},ns)
        for atom in list(expr.atoms(sp.Float)):
            expr = expr.subs(atom,sp.Float(f'{float(atom):.3g}'))
        return f'${sp.latex(expr,symbol_names=symbol_names)}$'
    except Exception:
        return eq_str

In [ ]:
with xr.open_dataset(os.path.join(SPLITSDIR,f'{SPLIT}.h5'),engine='h5netcdf') as ds:
    truetp = ds.tp.load()

with xr.open_dataset(os.path.join(SPLITSDIR,'norm_train.h5'),engine='h5netcdf') as ds:
    firstvar = next(iter(MODELS['nn']['runs'].values()))['fieldvars'][0]
    nsigs    = ds.sizes['sig'] if 'sig' in ds[firstvar].dims else 1

results = {}
for group in ['pod','nn','sr']:
    for name in MODELS[group]['runs']:
        if 'bl' in name or 'all' in name:
            filepath = os.path.join(PREDSDIR,f'{name}_{SPLIT}_predictions.nc')
            if not os.path.exists(filepath):
                continue
            with xr.open_dataset(filepath) as ds:
                predtp = ds.tp.load()
            if 'complexity' in predtp.dims:
                predtp = predtp.sel(complexity=PARETO_CS,drop=False)
            ytrue,ypred   = xr.align(truetp,predtp,join='inner')
            results[name] = (ytrue,ypred)

for name,eqspec in SRCONFIG.get('optimizedeqs',{}).items():
    filepath = os.path.join(PREDSDIR,f'{name}_{SPLIT}_predictions.nc')
    if not os.path.exists(filepath):
        continue
    with xr.open_dataset(filepath) as ds:
        predtp = ds.tp.load()
    ytrue,ypred   = xr.align(truetp,predtp,join='inner')
    results[name] = (ytrue,ypred)

print(f'Loaded {len(results)} models with predictions!')

In [ ]:
def get_r2(ytrue,ypred,dims=None):
    dims  = [d for d in ytrue.dims if d != 'seed'] if dims is None else dims
    ssres = ((ytrue-ypred)**2).sum(dim=dims,skipna=True)
    sstot = ((ytrue-ytrue.mean(dim=dims,skipna=True))**2).sum(dim=dims,skipna=True)
    return 1-ssres/sstot

def get_mean_std(x):
    return (float(x.mean('seed')),float(x.std('seed'))) if 'seed' in x.dims and x.sizes.get('seed',0)>1 else (float(x),0.0)

def collect_r2_records(results,r2_complexity=16):
    rows = []
    for name,(ytrue,ypred) in results.items():
        kind = get_kind(name)
        if kind == 'sr' and 'complexity' in ypred.dims:
            ypred = ypred.sel(complexity=r2_complexity)
            name  = f'{name}_c{r2_complexity}'
        r2mean,r2std = get_mean_std(get_r2(ytrue,ypred))
        color,_      = PLOTSTYLE.get(kind,('gray6','Unknown'))
        rows.append(dict(name=name,r2mean=r2mean,r2std=r2std,color=color,kind=kind))
    return sorted(rows,key=lambda r: r['r2mean'])

def collect_pareto_records(results):
    records,r2_half_mses = [],[]
    for name,(ytrue,ypred) in results.items():
        r2_half_mses.append(float(get_mse_at_r2(ytrue,r2_target=0.5)))
        kind = get_kind(name)

        if kind == 'sr' and 'complexity' in ypred.dims:
            for c in ypred.complexity.values:
                mse,_ = get_mean_std(get_mse(ytrue,ypred.sel(complexity=c)))
                records.append(dict(name=f'{name}_c{int(c)}',mse=mse,nparams=int(c),kind=kind))
            continue

        mse,_ = get_mean_std(get_mse(ytrue,ypred))
        if kind == 'pod':
            with np.load(os.path.join(MODELSDIR,'pod',f'{name}.npz')) as d:
                nparams = int(d['nparams'])
        elif kind == 'sr_opt':
            nparams = SRCONFIG.get('optimizedeqs',{}).get(name,{}).get('refcomplexity',10)
        elif kind == 'sr':
            csvpath = os.path.join(MODELSDIR,'sr',f'{name}_equations.csv')
            eqdf    = pd.read_csv(csvpath)
            nparams = int(eqdf.loc[eqdf['loss'].idxmin(),'complexity'])
        else:
            cfg     = MODELS['nn']['runs'][name]
            nparams = get_nn_complexity(cfg['kind'],len(cfg['fieldvars']),nsigs,len(cfg.get('localvars',[])))
        records.append(dict(name=name,mse=mse,nparams=nparams,kind=kind))
    return records,r2_half_mses

R2_LINES = [0.3,0.5]

def add_r2_lines(ax,ytrue):
    for r2 in R2_LINES:
        mval = float(get_mse_at_r2(ytrue,r2_target=r2))
        ax.axvline(mval,color='k',linestyle=':',lw=1)
        ax.text(mval,0.98,fr'$R^2>{r2}$',rotation=90,va='top',ha='right',
                transform=ax.get_xaxis_transform(),fontsize=7)

def plot_r2_and_pareto(results,r2_complexity=16):
    r2rows               = collect_r2_records(results,r2_complexity=r2_complexity)
    records,r2_half_mses = collect_pareto_records(results)
    any_seed             = any(r['r2std'] > 0 for r in r2rows)

    fig,axs = pplt.subplots(ncols=2,refwidth=3.6,refheight=3.0,share=False)

    ax     = axs[0]
    names  = [r['name'] for r in r2rows]
    means  = [r['r2mean'] for r in r2rows]
    stds   = [r['r2std'] for r in r2rows]
    colors = [r['color'] for r in r2rows]
    bars   = ax.barh(names,means,xerr=(stds if any_seed else None),color=colors,capsize=(3 if any_seed else 0))
    for bar,r2mean in zip(bars,means):
        ax.text(bar.get_width()-0.06,bar.get_y()+bar.get_height()/2,f'{r2mean:.3f}',va='center',ha='left',fontsize=7)
    ax.format(xlabel='R$^2$',ylabel='Model',grid=False,xminorticks='none')

    ax   = axs[1]
    seen = set()
    for rec in records:
        color,label = PLOTSTYLE.get(rec['kind'],('gray6',rec['kind']))
        if rec['kind'] == 'sr_opt':
            marker,s,zorder = 'D',70,5
        elif rec['kind'] == 'sr' and rec['nparams'] in PARETO_CS:
            marker,s,zorder = '*',90,4
        else:
            marker,s,zorder = 'o',35,3
        label = label if label not in seen else None
        ax.scatter(rec['mse'],rec['nparams'],color=color,marker=marker,s=s,zorder=zorder,label=label)
        seen.add(PLOTSTYLE.get(rec['kind'],('gray6',rec['kind']))[1])

    add_r2_lines(ax,next(iter(results.values()))[0])
    front = pareto_front(records)
    if len(front)>1:
        ax.plot([r['mse'] for r in front],[r['nparams'] for r in front],'k--',lw=1,label='Pareto Frontier')
    ax.format(xlabel='MSE (mm$^2$)',xlim=(1.25,3.5),ylabel='Model Complexity',yscale='log',yformatter='log')
    ax.legend(loc='r',ncols=1)
    pplt.show()
    return fig,axs

fig,axs = plot_r2_and_pareto(results,r2_complexity=16)
fig.save('fig_1.jpg')